## Step A3 : Fine-tune YOLOv8-nano

In [1]:
# Train YOLOv8-nano
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # nano = smallest, fastest

results = model.train(
    data="/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/dataset.yaml",
    epochs=50,
    imgsz=640,
    batch=16,          # auto-find max batch that fits in VRAM
    device=0,
    workers=8,         # parallel data loading
    amp=True,          # mixed precision → halves memory, faster training
    project="/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/runs/detect",
    name="yolov8n_cows",
    patience=10,       # stop early if no improvement for 10 epochs
    save=True,
    plots=True,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/oussema/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
New https://pypi.org/project/ultralytics/8.4.41 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.39 🚀 Python-3.13.12 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 7706MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics

/opt/miniconda3/lib/python3.13/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


MLflow: logging run_id(a34325d17f264c319ae9f3cc20d9f893) to runs/mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs/mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to /home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/runs/detect/yolov8n_cows-6
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/50      2.54G      1.534      1.266      1.219        235        640: 100% ━━━━━━━━━━━━ 882/882 6.5it/s 2:16<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 95/95 9.4it/s 10.1s<0.1s
                   all       3024      31955      0.903      0.846      0.937      0.599

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50      2.66G      1.288     0.7959      1.106        173        640: 100% ━━━━━━

## Step A3 : Evaluate training results

In [2]:
# Cell 16: Training summary + global loss
import pandas as pd

df_r = pd.read_csv("/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/runs/detect/yolov8n_cows-6/results.csv")
df_r.columns = df_r.columns.str.strip()

# Add global loss
df_r["train/global_loss"] = df_r["train/box_loss"] + df_r["train/cls_loss"] + df_r["train/dfl_loss"]
df_r["val/global_loss"]   = df_r["val/box_loss"]   + df_r["val/cls_loss"]   + df_r["val/dfl_loss"]

# Final epoch summary
last = df_r.iloc[-1]
print("=" * 40)
print("📊 FINAL RESULTS (epoch 50)")
print("=" * 40)
print(f"mAP@0.5:    {last['metrics/mAP50(B)']:.4f}  (target ≥ 0.80) ✅")
print(f"mAP@0.5-95: {last['metrics/mAP50-95(B)']:.4f}")
print(f"Precision:   {last['metrics/precision(B)']:.4f}")
print(f"Recall:      {last['metrics/recall(B)']:.4f}")
print(f"Train loss:  {last['train/global_loss']:.4f}")
print(f"Val loss:    {last['val/global_loss']:.4f}")

# Save updated CSV
df_r.to_csv("/home/oussema/mmcows-visual/runs/detect/yolov8n_cows/results.csv", index=False)
print("\n✅ Global loss column added to results.csv")

📊 FINAL RESULTS (epoch 50)
mAP@0.5:    0.9915  (target ≥ 0.80) ✅
mAP@0.5-95: 0.7880
Precision:   0.9705
Recall:      0.9695
Train loss:  2.0427
Val loss:    2.0944

✅ Global loss column added to results.csv


## Step A3 : Test set evaluation

In [3]:
# Cell 17: Evaluate on test set
from ultralytics import YOLO

model = YOLO("/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/runs/detect/yolov8n_cows-6/weights/best.pt")

metrics = model.val(
    data="/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/dataset.yaml",
    split="test",
    device=0,
)

print("=" * 40)
print("🧪 TEST SET RESULTS")
print("=" * 40)
print(f"mAP@0.5:    {metrics.box.map50:.4f}  (target ≥ 0.80)")
print(f"mAP@0.5-95: {metrics.box.map:.4f}")
print(f"Precision:   {metrics.box.mp:.4f}")
print(f"Recall:      {metrics.box.mr:.4f}")

Ultralytics 8.4.39 🚀 Python-3.13.12 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 7706MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 384.5±18.7 MB/s, size: 1151.1 KB)
val: Scanning /home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/labels/test... 3023 images, 156 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3023/3023 4.0Kit/s 0.8s0.1s
val: New cache created: /home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 189/189 7.9it/s 24.0s<0.2s
                   all       3023      32156      0.972      0.969      0.991      0.789
Speed: 0.4ms preprocess, 1.0ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to /home/oussema/mmcows-visual/notebooks/runs/detect/val
🧪 TEST SET RESULTS
mAP@0.5:    0.9914  (target ≥ 0.80)
mAP@0.5-95: 0.7886
Precision:   0.97

In [2]:
# Cell 18: Extract actual training hyperparameters
import yaml

args_path = "/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/runs/detect/yolov8n_cows-6/args.yaml"
with open(args_path) as f:
    args = yaml.safe_load(f)

keys = ['batch', 'epochs', 'imgsz', 'optimizer', 'lr0', 'lrf', 'momentum',
        'weight_decay', 'patience', 'amp', 'cache', 'workers', 'device',
        'cos_lr', 'warmup_epochs', 'warmup_momentum', 'close_mosaic']

for k in keys:
    print(f"{k}: {args.get(k, 'N/A')}")

batch: 16
epochs: 50
imgsz: 640
optimizer: auto
lr0: 0.01
lrf: 0.01
momentum: 0.937
weight_decay: 0.0005
patience: 10
amp: True
cache: False
workers: 8
device: 0
cos_lr: False
warmup_epochs: 3.0
warmup_momentum: 0.8
close_mosaic: 10


In [1]:
# Cell: Check loss weights
import yaml

hyp_path = "/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/runs/detect/yolov8n_cows-6/args.yaml"
with open(hyp_path) as f:
    args = yaml.safe_load(f)

for k in ['box', 'cls', 'dfl']:
    print(f"{k} weight: {args.get(k, 'N/A')}")

box weight: 7.5
cls weight: 0.5
dfl weight: 1.5


In [1]:
# Quick check before building report
import cv2
from pathlib import Path
img = cv2.imread(str(list(Path("/home/oussema/pfa-mmcows/store/data/clean/yolo_dataset/images/test").glob("cam_1_*.jpg"))[0]))
print(f"Image size: {img.shape[1]}x{img.shape[0]}")

Image size: 4480x2800
